<a href="https://colab.research.google.com/github/novalrnv/244107020210-mobile-course/blob/main/JS09/JS09-Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS09 - Approximate Nearest Neighbors (ANN)
## Tugas Lab 2 - Pencarian Track Terdekat pada Dataset Spotify

Percobaan exact NN, ANNOY, FAISS, dan HNSW pada dataset sekunder berukuran besar (Spotify Songs with Attributes and Lyrics).

**Langkah kerja:**
1. Download dataset dari Kaggle (pilih dataset **pertama** dari dua dataset), taruh satu folder dengan notebook ini.
2. Pilih fitur numerik saja, lalu normalisasi dengan `StandardScaler`.
3. Cari track terdekat dari satu track query.
4. Bandingkan exact NN, ANNOY, FAISS, dan HNSW (waktu, hasil tetangga, dan recall).

## Langkah 0 - Install dan Import Library

In [7]:
!pip install annoy faiss-cpu hnswlib

In [8]:
import glob
import time
import numpy as np
import pandas as pd
import faiss
import hnswlib
from annoy import AnnoyIndex
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

## Langkah 1 - Load Dataset

In [9]:
# Ganti DATA_PATH sesuai nama file CSV hasil download
DATA_PATH = '/content/songs_with_attributes_and_lyrics.csv'

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    found = glob.glob('*.csv')
    print('File tidak ditemukan, kandidat file csv di folder ini:', found)
    raise

print(df.shape)
df.head()

(955320, 17)


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,lyrics
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,HELLYEAH,0.415,0.6050,7,-11.157,1,0.0575,0.00116,0.838000,0.4710,0.193,100.059,79500.0,"He said he came from Jamaica,\n he owned a cou..."
1,2ASl4wirkeYm3OWZxXKYuq,!!,NaN,Yxngxr1,0.788,0.6480,7,-9.135,0,0.3150,0.90000,0.000000,0.1760,0.287,79.998,114000.0,"Fucked a bitch, now she running with my kids\n..."
2,69lcggVPmOr9cvPx9kLiiN,!!! - Interlude,Where I Belong EP,Glowie,0.000,0.0354,7,-20.151,0,0.0000,0.90800,0.000000,0.4790,0.000,0.000,11413.0,"Oh, my God, I'm going crazy\n"
3,4U7dlZjg1s9pjdppqZy0fm,!!De Repente!!,Un Palo Al Agua (20 Grandes Canciones),Rosendo,0.657,0.8820,5,-6.340,1,0.0385,0.00740,0.000013,0.0474,0.939,123.588,198173.0,Continuamente se extraña la gente si no puede ...
4,4v1IBp3Y3rpkWmWzIlkYju,!!De Repente!!,Fuera De Lugar,Rosendo,0.659,0.8930,5,-8.531,1,0.0411,0.09220,0.000019,0.0534,0.951,123.600,199827.0,Continuamente se extraña la gente si no puede ...


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 955320 entries, 0 to 955319
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                955320 non-null  object 
 1   name              955309 non-null  object 
 2   album_name        385557 non-null  object 
 3   artists           955315 non-null  object 
 4   danceability      955320 non-null  float64
 5   energy            955320 non-null  float64
 6   key               955320 non-null  object 
 7   loudness          955320 non-null  float64
 8   mode              955320 non-null  object 
 9   speechiness       955320 non-null  float64
 10  acousticness      955320 non-null  float64
 11  instrumentalness  955320 non-null  float64
 12  liveness          955320 non-null  float64
 13  valence           955320 non-null  float64
 14  tempo             955320 non-null  float64
 15  duration_ms       955320 non-null  float64
 16  lyrics            95

## Langkah 2 - Seleksi Fitur Numerik dan Normalisasi

Fitur yang dipakai sesuai boilerplate modul. Kalau ada nama kolom yang tidak ada di file kamu, kode di bawah akan memberi tahu dan hanya memakai kolom yang tersedia. Data dinormalisasi dengan `StandardScaler` karena skala tiap fitur berbeda (mis. `tempo` dan `loudness` jauh lebih besar dari `danceability`).

In [11]:
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

missing = [f for f in features if f not in df.columns]
if missing:
    print('Kolom tidak ditemukan, dilewati:', missing)
    print('Kolom numerik yang tersedia:', list(df.select_dtypes(include="number").columns))
features = [f for f in features if f in df.columns]

# buang baris yang punya nilai kosong pada fitur
df = df.dropna(subset=features).reset_index(drop=True)

X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

n, dim = X_scaled.shape
print(f'Jumlah track: {n}, jumlah fitur: {dim}')

k = 10  # jumlah nearest neighbors

Jumlah track: 955320, jumlah fitur: 9


## Langkah 3 - Pilih Track Query

Track query dipilih lewat indeks. Hasil pencarian akan menyertakan track itu sendiri di urutan pertama (jarak 0), jadi kita cari `k + 1` tetangga lalu membuang track query saat ditampilkan.

In [12]:
# Cari kolom nama track dan artis (nama kolom bisa berbeda tiap dataset)
def pick_col(cands):
    for cand in cands:
        if cand in df.columns:
            return cand
    return None

name_col = pick_col(['name', 'track_name', 'title', 'song'])
artist_col = pick_col(['artists', 'artist', 'track_artist', 'artist_name'])
print('Kolom nama:', name_col, '| Kolom artis:', artist_col)

def track_label(i):
    nm = df.loc[i, name_col] if name_col else f'track {i}'
    ar = df.loc[i, artist_col] if artist_col else ''
    return f'{nm} - {ar}' if ar else str(nm)

QUERY_IDX = 0   # ganti dengan indeks track lain jika ingin
query = X_scaled[QUERY_IDX].reshape(1, -1)
print('Track query:', track_label(QUERY_IDX))

K = k + 1  # +1 karena track query ikut terambil

Kolom nama: name | Kolom artis: artists
Track query: ! - HELLYEAH


## Langkah 4 - Exact NN (Brute Force)

In [13]:
t0 = time.time()
nn = NearestNeighbors(n_neighbors=K, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
build_exact = time.time() - t0

t0 = time.time()
dist_exact, idx_exact = nn.kneighbors(query)
time_exact = time.time() - t0
idx_exact = idx_exact[0]

print(f'Build : {build_exact:.4f} s')
print(f'Query : {time_exact*1000:.4f} ms')
print('Indeks tetangga:', idx_exact)

Build : 0.0803 s
Query : 270.1192 ms
Indeks tetangga: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698
  85956]


## Langkah 5 - ANNOY

In [14]:
t0 = time.time()
annoy_index = AnnoyIndex(dim, 'euclidean')
for i in range(n):
    annoy_index.add_item(i, X_scaled[i])
annoy_index.build(10)  # 10 trees
build_annoy = time.time() - t0

t0 = time.time()
idx_annoy = annoy_index.get_nns_by_vector(query[0], K)
time_annoy = time.time() - t0

print(f'Build : {build_annoy:.4f} s')
print(f'Query : {time_annoy*1000:.4f} ms')
print('Indeks tetangga:', idx_annoy)

Build : 17.8131 s
Query : 0.2937 ms
Indeks tetangga: [0, 764272, 749223, 625227, 833164, 676733, 523698, 85956, 888081, 763625, 906004]


## Langkah 6 - FAISS (IVFFlat)

In [15]:
nlist = max(4, int(np.sqrt(n)))   # jumlah cluster (inverted list)

t0 = time.time()
quantizer = faiss.IndexFlatL2(dim)
faiss_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)
faiss_index.train(X_scaled)
faiss_index.add(X_scaled)
build_faiss = time.time() - t0
faiss_index.nprobe = min(10, nlist)

t0 = time.time()
_, idx_faiss = faiss_index.search(query, K)
time_faiss = time.time() - t0
idx_faiss = idx_faiss[0]

print(f'Build : {build_faiss:.4f} s')
print(f'Query : {time_faiss*1000:.4f} ms')
print('Indeks tetangga:', idx_faiss)

Build : 1.9691 s
Query : 6.2494 ms
Indeks tetangga: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698
  85956]


## Langkah 7 - HNSW (hnswlib)

In [16]:
t0 = time.time()
hnsw_index = hnswlib.Index(space='l2', dim=dim)
hnsw_index.init_index(max_elements=n, ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
build_hnsw = time.time() - t0
hnsw_index.set_ef(50)

t0 = time.time()
labels, _ = hnsw_index.knn_query(query, k=K)
time_hnsw = time.time() - t0
idx_hnsw = labels[0]

print(f'Build : {build_hnsw:.4f} s')
print(f'Query : {time_hnsw*1000:.4f} ms')
print('Indeks tetangga:', idx_hnsw)

Build : 198.1116 s
Query : 0.5157 ms
Indeks tetangga: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698
  85956]


## Langkah 8 - Perbandingan Hasil Track Terdekat

Tabel berikut menampilkan 10 track terdekat (tanpa track query) menurut masing-masing metode.

In [17]:
def without_self(idx):
    return [i for i in idx if i != QUERY_IDX][:k]

rows = []
for rank in range(k):
    rows.append({
        'Rank': rank + 1,
        'Exact NN': track_label(without_self(idx_exact)[rank]),
        'ANNOY': track_label(without_self(idx_annoy)[rank]),
        'FAISS': track_label(without_self(idx_faiss)[rank]),
        'HNSW': track_label(without_self(idx_hnsw)[rank]),
    })

pd.set_option('display.max_colwidth', 45)
pd.DataFrame(rows)

,Rank,Exact NN,ANNOY,FAISS,HNSW
0,1,Intro (Who The Fuck Are You') - C-Lekktor,Testimonium - Lacrimosa,Intro (Who The Fuck Are You') - C-Lekktor,Intro (Who The Fuck Are You') - C-Lekktor
1,2,Testimonium - Lacrimosa,Sæglópur - Sigur Rós,Testimonium - Lacrimosa,Testimonium - Lacrimosa
2,3,Ultimate Satisfaction - The Field Remix -...,Púrpura - Margaritas Podridas,Ultimate Satisfaction - The Field Remix -...,Ultimate Satisfaction - The Field Remix -...
3,4,Sæglópur - Sigur Rós,Turn from Heaven - Therion,Sæglópur - Sigur Rós,Sæglópur - Sigur Rós
4,5,Autumn Interlude - Amethystium,See You When I Die - BOYO,Autumn Interlude - Amethystium,Autumn Interlude - Amethystium
5,6,Púrpura - Margaritas Podridas,Mrityu - Multani;Karun;Udbhav,Púrpura - Margaritas Podridas,Púrpura - Margaritas Podridas
6,7,Turn from Heaven - Therion,Bewitched - The Wedding Present,Turn from Heaven - Therion,Turn from Heaven - Therion
7,8,See You When I Die - BOYO,Who Makes the Nazis? - Live - Bonus Track...,See You When I Die - BOYO,See You When I Die - BOYO
8,9,Mrityu - Multani;Karun;Udbhav,Teri Yaadon Se - Lofi Flip - Jammy Weirdo...,Mrityu - Multani;Karun;Udbhav,Mrityu - Multani;Karun;Udbhav
9,10,Bewitched - The Wedding Present,Yngve-Frej - Torulf,Bewitched - The Wedding Present,Bewitched - The Wedding Present


## Langkah 9 - Benchmark Waktu dan Recall

Satu query saja kurang stabil untuk waktu, jadi dibuat benchmark pada 100 track query acak. **Recall@10** adalah persentase tetangga hasil exact NN yang berhasil ditemukan oleh ANN.

In [18]:
rng = np.random.RandomState(42)
q_ids = rng.choice(n, size=min(100, n), replace=False)

# Ground truth exact NN untuk semua query
_, gt = nn.kneighbors(X_scaled[q_ids])

def recall(I):
    return np.mean([len(set(I[i]) & set(gt[i])) / K for i in range(len(q_ids))])

# Exact
t0 = time.time()
for q in q_ids:
    nn.kneighbors(X_scaled[q].reshape(1, -1))
t_ex = (time.time() - t0) * 1000 / len(q_ids)

# ANNOY
t0 = time.time()
I_an = [annoy_index.get_nns_by_vector(X_scaled[q], K) for q in q_ids]
t_an = (time.time() - t0) * 1000 / len(q_ids)

# FAISS
t0 = time.time()
I_fa = [faiss_index.search(X_scaled[q].reshape(1, -1), K)[1][0] for q in q_ids]
t_fa = (time.time() - t0) * 1000 / len(q_ids)

# HNSW
t0 = time.time()
I_hn = [hnsw_index.knn_query(X_scaled[q].reshape(1, -1), k=K)[0][0] for q in q_ids]
t_hn = (time.time() - t0) * 1000 / len(q_ids)

hasil = pd.DataFrame({
    'Metode': ['Exact NN', 'ANNOY', 'FAISS', 'HNSW'],
    'Build (s)': [build_exact, build_annoy, build_faiss, build_hnsw],
    'Query (ms/query)': [t_ex, t_an, t_fa, t_hn],
    'Recall@10': [1.0, recall(I_an), recall(I_fa), recall(I_hn)],
})
hasil.round(4)

,Metode,Build (s),Query (ms/query),Recall@10
0,Exact NN,0.0803,54.3513,1.0000
1,ANNOY,17.8131,0.1978,0.8309
2,FAISS,1.9691,0.3823,0.9982
3,HNSW,198.1116,0.1197,0.9955


**Analisis:**

Exact NN menghasilkan recall sempurna (1.0), tapi query-nya paling lambat (54.35 ms/query). Ketiga metode ANN jauh lebih cepat, dengan recall yang berbeda-beda:

- HNSW: query tercepat (0.12 ms, sekitar 450x lebih cepat dari exact) dengan recall 0.9955, tapi build paling lama (198 detik).
- FAISS: recall tertinggi di antara ANN (0.9982), query 0.38 ms (sekitar 140x lebih cepat), dan build cuma 1.97 detik.
- ANNOY: query 0.20 ms (sekitar 275x lebih cepat), tapi recall paling rendah (0.8309), artinya sekitar 17% tetangga sebenarnya tidak ditemukan.